# All the import & installs needed for this


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install python-docx


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 15.5 MB/s eta 0:00:00


In [4]:
!pip install python-docx openpyxl

In [12]:
!pip install striprtf

# extract data from .rtf file made by Anymaze - this is for NOR


can delete #1
more notes

This worked in a usable way.  I used it for cohorts 2-5
I should re-export cohort 1 to make it look the same as the others

I need to go through which variables were not extracted b/c it was all text

Also, this yeilds all the data with no headers (variables).  bummer.  
but I can get all the data for each cohort from each sheet.


In [13]:
import re
import pandas as pd
import os
from striprtf.striprtf import rtf_to_text
from google.colab import drive

# 1. Connect to Drive
drive.mount('/content/drive')

# 2. Define your path
base_path = "/content/drive/Othercomputers/My Computer/Documents/Insulin/Behavior/data"
cohort_files = ["NOR-2.2.rtf", "NOR-2.3.rtf", "NOR-2.4.rtf", "NOR-2.5.rtf"]

def process_rtf_file(file_path):
    # Read the RTF file and convert to plain text
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
        text = rtf_to_text(content)

    # Split by the end of each data section
    sections = re.split(r'ANOVA\(note 3\).*?Notes:', text, flags=re.DOTALL)

    all_vars_dfs = {}

    for section in sections:
        section = section.strip()
        if not section or "Animal number" not in section:
            continue

        # Extract Variable Name
        var_match = re.search(r'^(.*?)\nAnimal number', section, flags=re.DOTALL)
        var_name = var_match.group(1).strip() if var_match else "Unknown_Var"

        # Clean Sheet Name
        sheet_name = re.sub(r'[\\/*?:\[\]]', '_', var_name)[:31]

        animal_map = {}
        # Find unique Animal IDs
        id_list = re.findall(r'^(\d{3,4})\s+', section, flags=re.MULTILINE)
        unique_ids = list(dict.fromkeys(id_list))

        for aid in unique_ids:
            # Capture numbers or n/a specifically tied to this Animal ID
            raw_values = re.findall(rf'(\d+\.\d+|\d+|n/a)\s*\({aid}\)', section)

            # Convert n/a to 0.0 and string numbers to floats
            cleaned_values = [0.0 if val.lower() == 'n/a' else float(val) for val in raw_values]

            if cleaned_values:
                animal_map[aid] = cleaned_values

        if animal_map:
            all_vars_dfs[sheet_name] = pd.DataFrame(dict([(k, pd.Series(v)) for k, v in animal_map.items()]))

    return all_vars_dfs

# --- MAIN LOOP ---
for filename in cohort_files:
    full_input_path = os.path.join(base_path, filename)

    if os.path.exists(full_input_path):
        print(f"Reading: {filename}...")

        try:
            extracted_data = process_rtf_file(full_input_path)

            output_filename = filename.replace(".rtf", "_Processed.xlsx")
            full_output_path = os.path.join(base_path, output_filename)

            with pd.ExcelWriter(full_output_path) as writer:
                for sheet, df in extracted_data.items():
                    df.to_excel(writer, sheet_name=sheet, index=False)

            print(f"✅ Created: {output_filename}")
        except Exception as e:
            print(f"❌ Error processing {filename}: {e}")
    else:
        print(f"❌ File not found: {full_input_path}")

print("\nProcessing complete!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Reading: NOR-2.2.rtf...
✅ Created: NOR-2.2_Processed.xlsx
Reading: NOR-2.3.rtf...
✅ Created: NOR-2.3_Processed.xlsx
Reading: NOR-2.4.rtf...
✅ Created: NOR-2.4_Processed.xlsx
Reading: NOR-2.5.rtf...
✅ Created: NOR-2.5_Processed.xlsx

Processing complete!


!pip install python-docx